# FCC BDC Jun 2025 — Fixed Broadband Availability download

Downloads the **Fixed Broadband Availability** dataset (all technologies, all states/territories) from `broadbandmap.fcc.gov` using the same public endpoints the data-download web UI calls. **No FCC API token / registration required.**

Run cells top-to-bottom. Stop after **Step 2** to inspect the file list before downloading; tweak the *filter* cell if you only want a subset.

Output goes to `data/fcc_bdc_jun2025/<state>/` (unzipped CSVs).

In [ ]:
import http.cookiejar, json, sys, time, urllib.parse, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

## Config — tweak as needed

In [ ]:
HOST = "https://broadbandmap.fcc.gov"
UA = "Mozilla/5.0 (telegrapher_ai bdc-fetch)"
OUT = Path("../data/fcc_bdc_jun2025")   # relative to scripts/ where this notebook lives
TARGET_FILING = "jun 2025"               # case-insensitive substring match
WORKERS = 8                              # parallel downloads (drop to 4 if you see 429s)
KEEP_ZIPS = False                        # True = keep raw .zip files in _zips/ after extract

OUT.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUT.resolve()}")

## HTTP session + helpers

Shared cookie jar so any session cookie set during the listing call is reused by the download endpoint.

In [ ]:
_jar = http.cookiejar.CookieJar()
_opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(_jar))
_opener.addheaders = [
    ("User-Agent", UA),
    ("Accept", "application/json, text/plain, */*"),
    ("Accept-Language", "en-US,en;q=0.9"),
    ("Referer", f"{HOST}/data-download/nationwide-data?version=jun2025&pubDataVer=jun2025"),
]

def warm_session():
    """Hit the data-download page once so the server sets any session cookies."""
    try:
        with _opener.open(
            f"{HOST}/data-download/nationwide-data?version=jun2025&pubDataVer=jun2025",
            timeout=60,
        ) as r:
            r.read(1)
    except Exception as e:
        print(f"  (warm_session: {e} -- continuing anyway)", file=sys.stderr)

def get_json(path, **params):
    url = f"{HOST}{path}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    with _opener.open(url, timeout=60) as r:
        return json.loads(r.read())

def stream(file_id, dest, expect_size=None, retries=4):
    url = f"{HOST}/nbm/map/api/getNBMDataDownloadFile/{file_id}/1"
    last = None
    for i in range(retries):
        try:
            with _opener.open(url, timeout=600) as r, open(dest, "wb") as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk: break
                    f.write(chunk)
            if expect_size and dest.stat().st_size != expect_size:
                raise IOError(f"size mismatch {dest.stat().st_size} vs {expect_size}")
            return
        except Exception as e:
            last = e
            time.sleep(2 ** i)
    raise last

warm_session()
print("session warmed")

## Step 1: Discover the Jun 2025 filing

Prints the full filings list so you can verify the match. If `TARGET_FILING` doesn't match, look at the JSON below and adjust the constant.

In [ ]:
filings = get_json("/nbm/map/api/published/filing")
print(json.dumps(filings, indent=2, default=str)[:3000])

seq = filings if isinstance(filings, list) else filings.get("data", filings)
filing = next((f for f in seq if TARGET_FILING in json.dumps(f, default=str).lower()), None)
assert filing is not None, f"No filing matched '{TARGET_FILING}' -- inspect JSON above and adjust"
uuid = filing.get("process_uuid") or filing.get("id") or filing.get("filing_id")
print(f"\nMatched filing: {filing}")
print(f"uuid = {uuid}")

## Step 2: List Fixed Broadband Availability files (all techs, all states)

Prints one sample record so you can sanity-check the field names (`file_id`, `file_name`, `file_size`, `state_name`/`state_abbr`). If anything is named differently, adjust the helper accessors in the download cell.

In [ ]:
listing = get_json(
    "/nbm/map/api/published/published_reports",
    process_uuid=uuid,
    data_type="availability",
    filing_subtype="fixed_broadband",
    technology_code=-1,   # -1 == "All Technologies" (verify in sample below)
)
files = listing.get("data", listing) if isinstance(listing, dict) else listing
assert isinstance(files, list), f"Unexpected listing shape: {json.dumps(listing, indent=2, default=str)[:2000]}"

print(f"Files returned: {len(files)}")
print(f"Total: {sum(int(f.get('file_size') or 0) for f in files) / 1e9:.2f} GB")
print("\nSample record:")
print(json.dumps(files[0], indent=2, default=str))

## Optional: filter to a subset

Skip this cell to download everything. Uncomment a line to pilot on one or a few states first (DC is smallest, ~50 MB).

In [ ]:
# files = [f for f in files if (f.get('state_abbr') or f.get('state_code') or '').upper() == 'DC']
# files = [f for f in files if (f.get('state_abbr') or f.get('state_code') or '').upper() in {'DC','DE','RI'}]

print(f"Will download {len(files)} files, total {sum(int(f.get('file_size') or 0) for f in files)/1e9:.2f} GB")

## Step 3: Download + extract

Parallel download with retry + size-check + resume. Each ZIP is extracted to `OUT/<state_name>/`. Re-running skips files already on disk at the right size.

In [ ]:
zips = OUT / "_zips"
zips.mkdir(exist_ok=True)

def fetch(meta):
    fid = meta.get("file_id") or meta.get("id")
    name = meta.get("file_name") or f"{fid}.zip"
    size = int(meta.get("file_size") or 0)
    zp = zips / name
    if not (zp.exists() and size and zp.stat().st_size == size):
        stream(fid, zp, expect_size=size or None)
    target_name = (meta.get("state_name") or meta.get("state_abbr") or "_unknown").replace(" ", "_")
    target = OUT / target_name
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(target)
    final_size = zp.stat().st_size if zp.exists() else size
    if not KEEP_ZIPS:
        zp.unlink()
    return f"{target_name} ({name}, {final_size} B)"

errors = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futs = {pool.submit(fetch, m): m for m in files}
    for fut in as_completed(futs):
        try:
            print(f"  ok  {fut.result()}")
        except Exception as e:
            m = futs[fut]
            msg = f"{m.get('state_name') or m.get('state_abbr')}: {e}"
            errors.append(msg)
            print(f"  ERR {msg}", file=sys.stderr)

print(f"\nDone. {len(files) - len(errors)}/{len(files)} succeeded.")

## Step 4: Write manifest

In [ ]:
(OUT / "manifest.json").write_text(json.dumps(
    {"filing": filing, "files": files, "errors": errors},
    indent=2,
    default=str,
))
print(f"Manifest: {(OUT / 'manifest.json').resolve()}")
print(f"Output:   {OUT.resolve()}")

# Quick directory listing
for p in sorted(OUT.iterdir()):
    if p.is_dir() and not p.name.startswith("_"):
        n_csv = len(list(p.glob("*.csv")))
        print(f"  {p.name:<40} {n_csv} csv")

## Quick peek at one CSV (sanity check)

In [ ]:
import pandas as pd

first_csv = next(
    (p for d in sorted(OUT.iterdir()) if d.is_dir() and not d.name.startswith("_")
     for p in d.glob("*.csv")),
    None,
)
if first_csv:
    print(f"Reading: {first_csv}")
    df = pd.read_csv(first_csv, nrows=5)
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")
    df
else:
    print("No CSVs found yet -- run the download cell first.")